## Benchmarking VectoreRAG + GraphRAG Recommnedation System

In [1]:
import json
from collections import defaultdict

# Load tool and workflow
with open("../../utilities/tools_metadata_downloader/data/galaxy_instance_tools_2025-12-04_23-58-00.json", "r") as f:
    tools = json.load(f)

with open("../../utilities/workflow_downloader/data/galaxy_iwc_workflows_20251205_162934.json", "r") as f:
    workflows = json.load(f)

print(f"Loaded {len(tools)} tools and {len(workflows)} workflows.")


Loaded 14923 tools and 20 workflows.


In [12]:
import os, sys
import sys
import os

# Set project root
project_root = os.path.abspath("../../")  # go up two levels from graphRAG
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(sys.path)  # confirm it now includes project root


# Now imports should work
from agents.graphRAG.pipeline.hybrid_rag_pipeline import HybridRAGPipeline
from agents.ingestion.Load.neo4j_client import Neo4jClient

# Define project root (adjust if notebook is inside subfolder)
project_root = os.path.abspath("../../")  # go two levels up from graphRAG
config_path = os.path.join(project_root, "agents/graphRAG/config/graph_db_config.yml")

# Initialize Neo4j client
neo_client = Neo4jClient(config_path=config_path)
hybrid_pipeline = HybridRAGPipeline(neo_client)



INFO:agents.ingestion.Load.neo4j_client:Connected to Neo4j at bolt://localhost:7687 as neo4j


['/home/henok/Desktop/projects/galaxy-agent-xp-II', '/home/henok/Desktop/projects/galaxy-agent-xp-II/agents/graphRAG', '/home/henok/.pyenv/versions/3.10.14/lib/python310.zip', '/home/henok/.pyenv/versions/3.10.14/lib/python3.10', '/home/henok/.pyenv/versions/3.10.14/lib/python3.10/lib-dynload', '', '/home/henok/Desktop/projects/galaxy-agent-xp-II/venv10/lib/python3.10/site-packages', '/tmp/tmpsq8_0e96']


INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (w:Workflow) ON (w.workflow_id)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (s:Step) ON (s.step_uid)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (t:Tool) ON (t.tool_id)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (c:Category) ON (c.category_id)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (i:ToolInput) ON (i.input_uid)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (o:ToolOutput) ON (o.output_uid)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (k:Keyword) ON (k.name)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (i:WorkflowInput) ON (i.input_id)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX 

 Loaded embedding model: BAAI/bge-base-en-v1.5
 Loaded embedding model: BAAI/bge-base-en-v1.5


In [17]:
import json
from collections import defaultdict

# Benchmarking function
def compute_hybrid_recall_at_k(pipeline, queries, ground_truth, k=5):
    """
    Compute recall@k for hybrid (Vector + Graph) RAG pipeline.
    
    Args:
        pipeline : your HybridRAGPipeline object
        queries : list of queries (strings)
        ground_truth : dict, query -> list of correct tool/workflow names
        k : top-k results to consider
    
    Returns:
        recall_scores : list of 0/1 per query
        avg_recall : float
    """
    recall_scores = []
    total_recall = 0

    for query in queries:
        # Run the hybrid pipeline
        result = pipeline.run(query=query, top_k=k)
        print(result)

        retrieved_items = []

        # Extract tools
        tools = result.get("tools", [])
        for t in tools[:k]:
            if isinstance(t, dict):
                retrieved_items.append(t.get("name", t.get("tool_name", "")))
            else:
                retrieved_items.append(str(t))

        # Extract workflows
        workflows = result.get("workflows", [])
        for wf in workflows[:k]:
            if isinstance(wf, dict):
                retrieved_items.append(wf.get("name", wf.get("workflow_name", "")))
            else:
                retrieved_items.append(str(wf))

        # Deduplicate
        retrieved_items = list(set(retrieved_items))

        # Compare with ground truth
        expected_items = ground_truth.get(query, [])
        match_found = any(gt in retrieved_items for gt in expected_items)
        recall = 1 if match_found else 0

        recall_scores.append(recall)
        total_recall += recall

        print(f"\nQuery: {query}")
        print(f"Expected: {expected_items}")
        print(f"Top-{k} retrieved: {retrieved_items}")
        print(f"Recall@{k}: {recall}")

    avg_recall = total_recall / len(queries)
    print(f"\n Average Recall@{k}: {avg_recall:.2f}")
    return recall_scores, avg_recall

In [24]:
queries = [
    # --- Workflow Queries ---
    "how to perform k-mer profiling for PacBio HiFi trio data for VGP2",
   
   
    # --- Tool Queries ---
    "I want to search a sequence database for a query sequence using jackhmmer"
    
   
]
   

ground_truth = {
    # --- Workflow Ground Truths ---
    "how to perform k-mer profiling for PacBio HiFi trio data for VGP2": ["kmer-profiling-hifi-trio-VGP2"],
    
    
    

    # --- Tool Ground Truths ---
    "I want to search a sequence database for a query sequence using jackhmmer": ["jackhmmer"]
   
   

}

recall_scores, avg_recall = compute_hybrid_recall_at_k(
        pipeline=hybrid_pipeline,
        queries=queries,
        ground_truth=ground_truth,
        k=5
    )


⚠️ Classification failed: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 31.502894972s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 31
}
]
Raw LLM output: 


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 30.82550808s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 30
}
]

In [ ]:
import json

# Load vague queries
with open("data/vague_queries.json") as f:
    vague_queries_dict = json.load(f)

# Flatten vague queries and create corresponding ground truth mapping
vague_query_list = []
vague_ground_truth = {}

for original_query, variations in vague_queries_dict.items():
    for variation in variations:
        vague_query_list.append(variation)
        vague_ground_truth[variation] = ground_truth[original_query]  # same ground truth

# Define a helper function to compute recall@k for HybridRAGPipeline
def compute_hybrid_recall_for_vague_queries(pipeline, queries, ground_truth, k=5):
    recall_scores = []
    for query in queries:
        # Run your HybridRAGPipeline
        result = pipeline.run(query=query, top_k=k, include_summary=False)
        
        # Extract names only
        retrieved_names = []
        if result["tools"]:
            retrieved_names.extend([t.get("name") or t.get("tool_name") for t in result["tools"]])
        if result["workflows"]:
            retrieved_names.extend([w.get("name") for w in result["workflows"]])

        # Check if any ground truth matches top-k
        expected = ground_truth.get(query, [])
        match_found = any(gt in retrieved_names[:k] for gt in expected)
        recall_scores.append(1 if match_found else 0)

    avg_recall = sum(recall_scores) / len(recall_scores)
    print(f"\n🔍 Average Recall@{k} for vague queries: {avg_recall:.2f}")
    return recall_scores, avg_recall

# Run recall benchmark
recall_scores, avg_recall = compute_hybrid_recall_for_vague_queries(
    pipeline=hybrid_pipeline,  # your HybridRAGPipeline instance
    queries=vague_query_list,
    ground_truth=vague_ground_truth,
    k=5
)


In [ ]:
# 10 Vague Queries for Tools
vague_tool_queries = [
    "I need to change my alignment location data format into simple regions.",
    "How do I figure out where the proteins are in my RNA sequences?",
    "Is there a way to standardize the names of the superbug genes I found?",
    "I want to look for specific chemical patterns in my database.",
    "How do I match my small RNA pieces to the transcript library?",
    "I have some region coordinates and I want them to look like alignment files.",
    "I need a fast way to map my reads to a reference.",
    "I have a bunch of separate files in a folder and I want them all in one single file.",
    "How can I group similar chemical signals from my mass spec data?",
    "I want to see if the bacteria in my two samples are basically the same or different."
]

# 10 Vague Queries for Workflows
vague_workflow_queries = [
    "I want to know if my bacteria are resistant to antibiotics.",
    "Can you help me build a complete genome starting from high-quality long reads?",
    "I want to organize my genomic pieces using chromosome folding data.",
    "How do I put together the DNA of the powerhouse of the cell?",
    "I need to test a lot of small molecules against a protein target on my computer.",
    "I just finished my assembly, how do I check if it's actually good and clean?",
    "I want to analyze the diversity of my microbial community from paired sequencing data.",
    "I have a new bacterial genome, how do I label all the genes in it?",
    "My assembly has too many extra copies of things, how do I get rid of the doubles?",
    "I want some graphs to show how long and complete my genome scaffolds are."
]

# Combined Ground Truth Mapping
vague_ground_truth = {
    # Tools
    "I need to change my alignment location data format into simple regions.": ["MAF to Interval"],
    "How do I figure out where the proteins are in my RNA sequences?": ["TransDecoder"],
    "Is there a way to standardize the names of the superbug genes I found?": ["argNorm"],
    "I want to look for specific chemical patterns in my database.": ["Substructure Search"],
    "How do I match my small RNA pieces to the transcript library?": ["ChiRA map"],
    "I have some region coordinates and I want them to look like alignment files.": ["Convert from BED to BAM"],
    "I need a fast way to map my reads to a reference.": ["HISAT2"],
    "I have a bunch of separate files in a folder and I want them all in one single file.": ["Collapse Collection"],
    "How can I group similar chemical signals from my mass spec data?": ["RAMClustR"],
    "I want to see if the bacteria in my two samples are basically the same or different.": ["Parsimony"],

    # Workflows
    "I want to know if my bacteria are resistant to antibiotics.": ["AMR Gene Detection"],
    "Can you help me build a complete genome starting from high-quality long reads?": ["Genome Assembly from Hifi reads - VGP3"],
    "I want to organize my genomic pieces using chromosome folding data.": ["Scaffolding with Hi-C data VGP8"],
    "How do I put together the DNA of the powerhouse of the cell?": ["Mitogenome Assembly VGP0"],
    "I need to test a lot of small molecules against a protein target on my computer.": ["Fragment-based virtual screening using rDock for docking and SuCOS for pose scoring"],
    "I just finished my assembly, how do I check if it's actually good and clean?": ["Post-Assembly Quality Control and Contamination Check for Bacterial Genomes"],
    "I want to analyze the diversity of my microbial community from paired sequencing data.": ["dada2 amplicon analysis pipeline - for paired end data"],
    "I have a new bacterial genome, how do I label all the genes in it?": ["bacterial_genome_annotation"],
    "My assembly has too many extra copies of things, how do I get rid of the doubles?": ["Purge duplicate contigs from a diploid assembly VGP6"],
    "I want some graphs to show how long and complete my genome scaffolds are.": ["Generate Nx and Size plots for multiple assemblies"]
}

In [ ]:
# Combine all queries
vague_queries = vague_tool_queries + vague_workflow_queries

# Helper function for Recall@k
def compute_hybrid_recall_vague(pipeline, queries, ground_truth, k=5):
    recall_scores = []
    for query in queries:
        # Run your HybridRAGPipeline
        result = pipeline.run(query=query, top_k=k, include_summary=False)
        
        # Extract only tool/workflow names
        retrieved_names = []
        if result["tools"]:
            retrieved_names.extend([t.get("name") or t.get("tool_name") for t in result["tools"]])
        if result["workflows"]:
            retrieved_names.extend([w.get("name") for w in result["workflows"]])

        # Check for at least one match in top-k
        expected = ground_truth.get(query, [])
        match_found = any(gt in retrieved_names[:k] for gt in expected)
        recall_scores.append(1 if match_found else 0)

        print(f"\nQuery: {query}")
        print(f"Expected: {expected}")
        print(f"Retrieved (top-{k}): {retrieved_names[:k]}")
        print(f"Recall@{k}: {1 if match_found else 0}")

    avg_recall = sum(recall_scores) / len(recall_scores)
    print(f"\n Average Recall@{k} for vague queries: {avg_recall:.2f}")
    return recall_scores, avg_recall

# Run benchmark on vague queries
recall_scores, avg_recall = compute_hybrid_recall_vague(
    pipeline=hybrid_pipeline,  
    queries=vague_queries,
    ground_truth=vague_ground_truth,
    k=5
)